# Reto: Spaceship Titanic

### Inteligencia Artificial Avanzada para la Ciencia de Datos

---

**Integrantes**

- Ana Paula Moreno
- David Tinoco Romero
- Emilio Páez de la Mora
- Alejandro Vázquez
- Emilio Torres

---

### Tecnológico de Monterrey

---

## Objetivo del notebook

Este notebook consolida el trabajo de los tres avances del reto **Spaceship
Titanic** en un solo documento de referencia: preprocesamiento y feature
engineering (Avance 1), selección de familia de modelo y exploración inicial
de hiperparámetros (Avance 2), y diagnóstico, tuning fino, componente
individual y decisión final (Avance 3).

El objetivo **no es corregir ni repetir** el trabajo ya hecho, sino
**justificar de forma consolidada** las decisiones que llevaron al modelo
final, con las gráficas y el razonamiento más importantes de cada etapa.
Para el detalle completo de cada análisis (EDA exhaustivo, todas las
corridas de búsqueda de hiperparámetros, las 5 subsecciones individuales
completas, etc.), este notebook cita el avance correspondiente — el detalle
íntegro vive en `Notebooks/`, dentro de este mismo repositorio.

Al final, el notebook entrena la corrida óptima de los dos modelos que el
equipo dejó como estrategia final (Random Forest tuneado y SVC tuneado) y
los exporta con `joblib`, junto con el preprocesador, para alimentar
directamente la interfaz/dashboard de la siguiente etapa del reto.

**Nota sobre reproducibilidad:** a diferencia de los avances individuales,
aquí no se ejecutan búsquedas de hiperparámetros completas (`GridSearchCV`/
`RandomizedSearchCV`) — se usan directamente los mejores hiperparámetros ya
encontrados y documentados en los avances correspondientes, para que este
notebook corra de principio a fin en minutos, no en horas.

Este notebook está organizado en las siguientes secciones:

1. Carga de Datos
2. Preprocesamiento y Feature Engineering
3. Selección de Familia de Modelo
4. Diagnóstico y Tuning del Modelo Base
5. Resultados del Componente Individual
6. Modelo Final — Estrategia Dual (SVC + Random Forest)
7. Exportación de Modelos
8. Conclusión Final del Equipo
9. Apéndice A — Deck como Variable Ordinal
10. Apéndice B — Verificación de Varianza por `group_size`

# 1. Carga de Datos

Partimos de los datos crudos de Kaggle (`train.csv`/`test.csv`), igual que
en el Avance 1, para que este notebook sea autocontenido y no dependa de
archivos intermedios que no estén en el repositorio.

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("..") / "data"
RANDOM_STATE = 42

df_train = pd.read_csv(DATA / "train.csv")
df_test = pd.read_csv(DATA / "test.csv")

print("train:", df_train.shape)
print("test: ", df_test.shape)
df_train.head()

train: (8693, 14)
test:  (4277, 13)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


# 2. Preprocesamiento y Feature Engineering

Consolidamos aquí el feature engineering completo del Avance 1
(`Notebooks/Reto_Primer_Avance_Spaceship_Titanic.ipynb`, Secciones 3 a 7),
encapsulado como una clase (`SpaceshipPreprocessor`) que implementa la
interfaz `fit`/`transform` de scikit-learn. Esto cumple con la recomendación
del profesor de usar objetos, y además hace que el mismo preprocesamiento
se pueda reutilizar tanto aquí como en la interfaz/dashboard final, sin
duplicar lógica.

**Diferencia clave respecto al Avance 1:** las estadísticas de imputación
(medianas, modas) y el escalador se **ajustan una sola vez sobre `train`**
(`fit`) y luego se **aplican tal cual** sobre cualquier dato nuevo
(`transform`) — incluyendo test.csv y, más adelante, pasajeros individuales
en el dashboard. En el notebook original estas estadísticas se recalculaban
sobre el mismo dataframe que se procesaba, lo cual es correcto para un
análisis de una sola pasada pero no es viable para predecir sobre datos
nuevos uno a la vez.

Para el detalle completo del EDA que justifica cada decisión (por qué se
descartan `VIP`/`Name`, por qué `log1p`+`RobustScaler`, por qué bins de 300
en `CabinNum`, etc.), ver Avance 1, Secciones 2 a 7. Aquí solo se retoma el
resultado ya justificado.

In [8]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import RobustScaler

SPEND_COLS = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

class SpaceshipPreprocessor(BaseEstimator, TransformerMixin):
    """
    Replica el feature engineering del Avance 1 (Secciones 3-7) como un
    objeto reusable. fit() aprende estadísticas SOLO de los datos de
    entrenamiento; transform() las aplica a cualquier dato nuevo.
    """

    def __init__(self):
        self.bins_cabin = [-np.inf, 299, 599, 899, 1199, 1499, np.inf]
        self.labels_cabin = ["0-299", "300-599", "600-899", "900-1199", "1200-1499", "1500+"]

    def fit(self, X, y=None):
        df = X.copy()
        df = df.drop(columns=["Name", "VIP"], errors="ignore")

        df["Deck"] = df["Cabin"].str.split("/").str[0]
        df["CabinNum"] = pd.to_numeric(df["Cabin"].str.split("/").str[1], errors="coerce")
        df["Side"] = df["Cabin"].str.split("/").str[2]
        gastos_totales = df[SPEND_COLS].sum(axis=1)
        df.loc[df["CryoSleep"].isnull() & (gastos_totales > 0), "CryoSleep"] = False

        self.medianas_ = {col: df[col].median() for col in ["Age", "CabinNum"] + SPEND_COLS}
        self.modas_ = {col: df[col].mode()[0] for col in ["HomePlanet", "Destination", "CryoSleep", "Deck", "Side"]}

        df_imputado = self._imputar(df)
        df_codificado = self._codificar(df_imputado)
        self.cols_to_scale_ = ["Age", "group_size"] + SPEND_COLS
        self.scaler_ = RobustScaler()
        self.scaler_.fit(df_codificado[self.cols_to_scale_])

        self.columnas_finales_ = self._escalar(df_codificado.copy()).drop(columns=["Transported"], errors="ignore").columns

        return self

    def transform(self, X):
        df = X.copy()
        df = df.drop(columns=["Name", "VIP"], errors="ignore")

        df["Deck"] = df["Cabin"].str.split("/").str[0]
        df["CabinNum"] = pd.to_numeric(df["Cabin"].str.split("/").str[1], errors="coerce")
        df["Side"] = df["Cabin"].str.split("/").str[2]
        gastos_totales = df[SPEND_COLS].sum(axis=1)
        df.loc[df["CryoSleep"].isnull() & (gastos_totales > 0), "CryoSleep"] = False

        df = self._imputar(df)
        df = self._codificar(df)
        df = self._escalar(df)

        target = df["Transported"] if "Transported" in df.columns else None
        df = df.reindex(columns=self.columnas_finales_, fill_value=0)
        if target is not None:
            df["Transported"] = target

        return df

    def _imputar(self, df):
        for col in SPEND_COLS:
            df.loc[(df["CryoSleep"] == True) & (df[col].isnull()), col] = 0.0
        for col in ["Age", "CabinNum"] + SPEND_COLS:
            df[col] = df[col].fillna(self.medianas_[col])
        for col in ["HomePlanet", "Destination", "CryoSleep", "Deck", "Side"]:
            df[col] = df[col].fillna(self.modas_[col])
        return df

    def _codificar(self, df):
        df["group"] = df["PassengerId"].str.split("_").str[0]
        df["group_size"] = df["group"].map(df["group"].value_counts())
        df.drop(columns=["Cabin", "PassengerId", "group"], inplace=True, errors="ignore")

        df["CabinNumBin"] = pd.cut(df["CabinNum"], bins=self.bins_cabin, labels=self.labels_cabin)
        df.drop(columns=["CabinNum"], inplace=True)

        df["HasSpent"] = (df[SPEND_COLS].sum(axis=1) > 0).astype(int)
        df["CryoSleep"] = df["CryoSleep"].astype(int)
        df["Side"] = df["Side"].map({"P": 0, "S": 1})
        if "Transported" in df.columns:
            df["Transported"] = df["Transported"].astype(int)

        df = pd.get_dummies(df, columns=["HomePlanet", "Destination", "Deck", "CabinNumBin"], drop_first=True, dtype=int)
        return df

    def _escalar(self, df):
        for col in SPEND_COLS:
            df[col] = np.log1p(df[col])
        df[self.cols_to_scale_] = self.scaler_.transform(df[self.cols_to_scale_])
        return df

In [9]:
preprocesador = SpaceshipPreprocessor()
preprocesador.fit(df_train)

df_prep = preprocesador.transform(df_train)

print("Shape df_prep:", df_prep.shape)
print("Nulos totales:", df_prep.isnull().sum().sum())
df_prep.head()

Shape df_prep: (8693, 27)
Nulos totales: 0


,CryoSleep,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Side,group_size,HasSpent,...,Deck_E,Deck_F,Deck_G,Deck_T,CabinNumBin_300-599,CabinNumBin_600-899,CabinNumBin_900-1199,CabinNumBin_1200-1499,CabinNumBin_1500+,Transported
0,0,0.705882,0.000000,0.000000,0.000000,0.000000,0.000000,0,0.0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,-0.176471,0.114646,0.037747,0.148095,0.119055,0.095167,1,0.0,1,...,0,1,0,0,0,0,0,0,0,1
2,0,1.823529,0.092297,0.134136,0.000000,0.166269,0.097801,1,0.5,1,...,0,0,0,0,0,0,0,0,0,0
3,0,0.352941,0.000000,0.117340,0.269041,0.153033,0.131696,1,0.5,1,...,0,0,0,0,0,0,0,0,0,0
4,0,-0.647059,0.139440,0.069880,0.228358,0.119596,0.027465,1,0.0,1,...,0,1,0,0,0,0,0,0,0,1


# 3. Selección de Familia de Modelo

Antes de tunear nada, el Avance 2 partió de dos preguntas: ¿qué métricas
tienen sentido para este problema?, y ¿qué familia de modelo conviene
profundizar? Aquí resumimos ese proceso; el detalle completo (incluyendo la
exploración individual de hiperparámetros de cada integrante) está en
`Notebooks/Reto_Segundo_Avance.ipynb`, Secciones 2 a 5.

**Métricas elegidas:** con las clases prácticamente balanceadas (~50.4% /
49.6%), Accuracy es una métrica confiable como referencia general. Se
complementa con Precision, Recall y F1-score porque, aunque Accuracy resume
el desempeño global, no distingue qué tipo de error comete el modelo — y en
este problema no hay una razón de negocio para preferir evitar falsos
positivos sobre falsos negativos, así que F1 (que balancea ambos) es la
métrica compuesta de referencia.

In [11]:
from sklearn.model_selection import train_test_split

X = df_prep.drop(columns=["Transported"])
y = df_prep["Transported"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

clase_mayoritaria = y.mode()[0]
baseline_accuracy = (y == clase_mayoritaria).mean()

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print(f"Baseline (accuracy): {baseline_accuracy:.4f}")

X_train: (6954, 26) X_test: (1739, 26)
Baseline (accuracy): 0.5036


**Baseline:** predecir siempre la clase mayoritaria da un accuracy de
~0.5036 — cualquier modelo debe superar esto claramente para justificar su
uso.

**Dos modelos de familias distintas** para tener un punto de comparación
real antes de profundizar en uno solo:

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

modelo_lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
modelo_lr.fit(X_train, y_train)
y_pred_lr = modelo_lr.predict(X_test)

modelo_rf_base = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=RANDOM_STATE)
modelo_rf_base.fit(X_train, y_train)
y_pred_rf_base = modelo_rf_base.predict(X_test)

resultados = pd.DataFrame({
    "Modelo": ["Baseline", "Regresión Logística", "Random Forest"],
    "Accuracy": [baseline_accuracy, accuracy_score(y_test, y_pred_lr), accuracy_score(y_test, y_pred_rf_base)],
    "Precision": [None, precision_score(y_test, y_pred_lr), precision_score(y_test, y_pred_rf_base)],
    "Recall": [None, recall_score(y_test, y_pred_lr), recall_score(y_test, y_pred_rf_base)],
    "F1-score": [None, f1_score(y_test, y_pred_lr), f1_score(y_test, y_pred_rf_base)],
})
resultados

,Modelo,Accuracy,Precision,Recall,F1-score
0,Baseline,0.503624,NaN,NaN,NaN
1,Regresión Logística,0.784359,0.796450,0.768265,0.782103
2,Random Forest,0.798735,0.800915,0.799087,0.800000


## Elección de familia de modelo

Random Forest superó a Regresión Logística en Accuracy (+0.0052), Precision
(+0.0140) y F1-score (+0.0020), pero perdió en Recall (−0.0103) — una
mejora pequeña, no aplastante. Se eligió **Random Forest** no porque "gane
en todo", sino porque fue la única métrica compuesta (F1) donde hubo mejora
consistente, y porque un ensamble de árboles tiene más margen de mejora vía
tuning de hiperparámetros que una Regresión Logística.

La exploración individual de hiperparámetros del Avance 2 (Sección 5 — cada
integrante varió un hiperparámetro distinto de Random Forest:
`min_samples_leaf`, `max_depth`, `min_samples_split`, `n_estimators`)
confirmó que ningún cambio aislado producía una mejora clara, lo cual
motivó el diagnóstico más riguroso (curvas de aprendizaje/validación) que
se hizo en el Avance 3, Sección 1 — ver Sección 4 de este notebook.